# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EimanZahra1472/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
!git clone https://github.com/EimanZahra1472/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
print("Ready.")

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 137 (delta 46), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 1.88 MiB | 8.70 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Ready.


In [2]:
q = f"""
SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 5
"""
sample = con.sql(q).df()
print(sample.columns.tolist())
sample

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


One row = one (client, content item, report date) triple in fact_content_daily_performance one page's daily performance snapshot for one client, on one day. Time window: month=2026-03, a mid-panel month, chosen for feature and label development. I avoided the _sample table (final month, June 2026) throughout, since per this card's warning it's the natural outcome window for any past→future label and would leak the future into feature-building.

Verified: COUNT(*) = COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) = 9,841,378 — no duplicate rows per key, confirming the grain claim exactly.

In [4]:
q = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_keys
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(q).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_keys
0,9841378,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features used: gsc_impressions, gsc_avg_position daily observed search signals, known the moment that day's report lands.
Feature candidates not used here: gsc_clicks, ga4_engaged_sessions, scroll_events  clicks was excluded from the honest model because it's what CTR (and therefore the label) is built from; GA4 and scroll fields were mostly <NA> in this slice (see limits below).
Label: low_ctr_label = (ctr == 0), i.e. did this page get zero clicks despite having impressions. Built for the leakage demonstration only  not a validated capstone target yet.
Context / join keys: client_hash_id, content_hash_id, report_date, month used for grouping and filtering only, never as model features.
Excluded: ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, sessions_ai excluded because AI-session data is extremely sparse (~30K of 78.8M rows warehouse-wide per the lane guide); including them here would mostly add missing-value noise, not signal.

In [5]:
q = f"""
SELECT
  COUNT(*) AS rows,
  MIN(report_date) AS min_date,
  MAX(report_date) AS max_date,
  COUNT(DISTINCT client_hash_id) AS n_clients,
  COUNT(DISTINCT content_hash_id) AS n_content_items
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(q).df()

,rows,min_date,max_date,n_clients,n_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q = f"""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
  SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(q).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061.0,413966.0


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q = f"""
SELECT
  report_date, client_hash_id, content_hash_id,
  gsc_impressions,
  gsc_clicks,
  CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END AS ctr,
  gsc_avg_position,
  ga4_engaged_sessions
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
LIMIT 2000
"""
feat_df = con.sql(q).df()
print(feat_df.shape)
feat_df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(2000, 8)


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,ga4_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,0.000000,3.350000,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,0.000000,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,0.008000,4.928000,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,0.000000,4.000000,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,0.000000,2.272727,<NA>
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,0.004184,7.347280,<NA>
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,0.000000,7.832461,<NA>
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,0.000000,3.272727,<NA>
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,0.000000,5.636364,<NA>
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,0.000000,4.500000,<NA>


In [9]:
print(feat_df["ctr"].describe())
print(feat_df["ctr"].value_counts().head(10))

count    2000.000000
mean        0.003319
std         0.041003
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: ctr, dtype: float64
ctr
0.000000    1740
1.000000       3
0.007692       3
0.047619       3
0.002994       2
0.041667       2
0.002778       2
0.012500       2
0.029412       2
0.009174       2
Name: count, dtype: int64


In [10]:
feat_df_v2 = feat_df[feat_df["gsc_impressions"] >= 50].copy()
print(feat_df_v2["ctr"].describe())
print(feat_df_v2.shape)

count    666.000000
mean       0.002570
std        0.004831
min        0.000000
25%        0.000000
50%        0.000000
75%        0.003489
max        0.037037
Name: ctr, dtype: float64
(666, 9)


In [12]:
feat_df_v2["low_ctr_label"] = (feat_df_v2["ctr"] == 0).astype(int)
print(feat_df_v2["low_ctr_label"].value_counts())

low_ctr_label
1    435
0    231
Name: count, dtype: int64


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_honest = feat_df_v2[["gsc_impressions", "gsc_avg_position"]].fillna(0)
y = feat_df_v2["low_ctr_label"]

honest_model = LogisticRegression(max_iter=1000).fit(X_honest, y)
honest_auc = roc_auc_score(y, honest_model.predict_proba(X_honest)[:,1])
print(f"Honest AUC (no leak): {honest_auc:.3f}")

X_leaky = X_honest.copy()
X_leaky["ctr_bucket_leak"] = feat_df_v2["ctr"]

leaky_model = LogisticRegression(max_iter=1000).fit(X_leaky, y)
leaky_auc = roc_auc_score(y, leaky_model.predict_proba(X_leaky)[:,1])
print(f"Leaky AUC (ctr fed in directly): {leaky_auc:.3f}  <- jumps toward 1.0, because ctr IS the label")

print(f"\nFinal honest AUC kept: {honest_auc:.3f}")

Honest AUC (no leak): 0.800
Leaky AUC (ctr fed in directly): 0.802  <- jumps toward 1.0, because ctr IS the label

Final honest AUC kept: 0.800


In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_leaky_scaled = scaler.fit_transform(X_leaky)

leaky_model_v2 = LogisticRegression(max_iter=1000).fit(X_leaky_scaled, y)
leaky_auc_v2 = roc_auc_score(y, leaky_model_v2.predict_proba(X_leaky_scaled)[:,1])
print(f"Leaky AUC (scaled features): {leaky_auc_v2:.3f}")

Leaky AUC (scaled features): 0.999


Query 1 — grain check: total_rows == distinct_keys == 9,841,378. Confirms one row per (client, content, date), no duplicates.

Query 2 — row count and date span: 9,841,378 rows spanning 2026-03-01 to 2026-03-31, across 55 clients and 331,437 distinct content items.

Query 3 — availability (IS TRUE): of 9,841,378 total rows, 3,611,061 have gsc_data_available IS TRUE and only 413,966 have ga4_data_available IS TRUE. This confirms GA4 coverage is far thinner than GSC coverage in this month most rows are search-only.

Five features (built from a 2,000-row sample filtered to gsc_data_available IS TRUE AND gsc_impressions > 0):

gsc_impressions  available at the decision moment: it's that day's own observed search impression count.
gsc_clicks same-day observed clicks, known as soon as the report lands.
ctr (computed as clicks/impressions) derived purely from that same day's numbers, no future data involved.
gsc_avg_position  the day's observed average search position.
ga4_engaged_sessions  same-day observed engagement count, though mostly missing in this slice due to GA4 coverage gaps (see limits).

The leakage trap: I built a proxy label, low_ctr_label = (ctr == 0), on a volume-filtered subset (gsc_impressions >= 50, n=666; 435 zero-click pages, 231 with clicks). A logistic regression trained honestly on gsc_impressions and gsc_avg_position scored AUC = 0.800. I then deliberately added ctr itself back in as a third feature  the exact column the label was derived from. Unscaled, the leak barely moved the score (0.800 → 0.802), because logistic regression's regularization underweighted the tiny-scale ctr column against the much larger gsc_impressions values. Once I standardized the features, the leak showed its true size: AUC jumped to 0.999. This was an important lesson beyond the original notebook 02 leakage demo  leakage doesn't always announce itself loudly; unscaled features can mask it, so scaling matters even for a leakage audit, not just for model performance. I deleted the leaked feature and kept the honest AUC of 0.800 as the real number

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [15]:
q = f"""
SELECT
  ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_ga4_available
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(q).df()

,pct_ga4_available
0,4.2


Unbalanced GA4 coverage: only 413,966 of 9,841,378 rows (4.2%) have ga4_data_available IS TRUE this month  most clients in this slice are GSC-only, so any GA4-based feature (engagement, sessions) will be missing for the majority of rows, not genuinely zero. Treating that missingness as "no engagement" would be a measurement error, not a real observation.
Sparse click signal at low volume: in an unfiltered sample, 1,740 of 2,000 rows (87%) had ctr == 0, meaning median-based label splits are frequently degenerate at low impression volumes. Any CTR-based label needs a minimum-impressions filter to avoid being dominated by near-empty pages.
Single-month slice: this contract only covers March 2026. Client history is an unbalanced panel warehouse-wide (per the lane guide, only 9 of 70 clients have 12+ months of history), so seasonality or trend-based features built from a single month can't be validated for stability across time from this slice alone.
No causal claims possible: this data can show associations (e.g. low CTR correlating with position or impressions) but never that changing a page caused a CTR change  that would require an experiment this dataset doesn't support

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.